In [3]:
import copy
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from shapely.affinity import affine_transform
from shapely.geometry import Point, LineString, Polygon, MultiPoint, MultiLineString, MultiPolygon, box
from geopandas import GeoSeries, GeoDataFrame
from shapely.ops import split, unary_union, polygonize
import pickle

/home/lxh/anaconda3/envs/urban/lib/python3.8/site-packages/geopandas/_compat.py:112: UserWarning: The Shapely GEOS version (3.10.2-CAPI-1.16.0) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(


In [3]:
gdf=gpd.read_file('rd.geojson')
gdf.crs = None

In [4]:
gdf.explore()

In [4]:
def find_max_coordinates(gdf):
    # Get the maximum x and y coordinates
    max_x = gdf.geometry.x.max()
    max_y = gdf.geometry.y.max()
    
    return max_x, max_y
find_max_coordinates(gdf[gdf['type']==13])

NameError: name 'gdf' is not defined

In [12]:
NON_BLOCK_LAND_USE = (
    'outside',
    'feasible',
    'road',
    'boundary'
)

BLOCK_LAND_USE = (
    'residential',
    'business',
    'business_h',
    'green_l',
    'residential_h',
    'school',
    'hospital_l',
    'hospital_s',
    'recreation'
)

LAND_USE = (
    NON_BLOCK_LAND_USE + BLOCK_LAND_USE)

OUTSIDE = 0
FEASIBLE = 1
ROAD = 2
BOUNDARY = 3
RESIDENTIAL = 4
BUSINESS = 5
BUSINESS_H = 6
GREEN_L = 7
RESIDENTIAL_H = 8
SCHOOL = 9
HOSPITAL_L = 10
HOSPITAL_S = 11
RECREATION = 12


LAND_USE_ID = (
    OUTSIDE,
    FEASIBLE,
    ROAD,
    BOUNDARY,
    RESIDENTIAL,
    BUSINESS,
    BUSINESS_H,
    GREEN_L,
    RESIDENTIAL_H,
    SCHOOL,
    HOSPITAL_L,
    HOSPITAL_S,
    RECREATION,
)

BLOCK_LAND_TYPE = (
    RESIDENTIAL,
    BUSINESS,
    BUSINESS_H,
    GREEN_L,
    RESIDENTIAL_H,
    SCHOOL,
    HOSPITAL_L,
    HOSPITAL_S,
    RECREATION,
)

NUM_TYPES = len(LAND_USE_ID)

LAND_USE_ID_MAP = dict(
    zip(LAND_USE, LAND_USE_ID))

LAND_USE_ID_MAP_INV = dict(
    zip(LAND_USE_ID, LAND_USE))

INTERSECTION = 13

RESIDENTIAL_ID = (
    RESIDENTIAL,
    RESIDENTIAL_H
)

PUBLIC_SERVICES_ID = (
    BUSINESS,
    BUSINESS_H,
    SCHOOL,
    (HOSPITAL_L, HOSPITAL_S),
    RECREATION
)

PUBLIC_SERVICES = (
    'shopping',
    'working',
    'education',
    'medical care',
    'entertainment'
)

GREEN_ID = (
    GREEN_L,
    RECREATION
)
GREEN_AREA_THRESHOLD = 2000

TYPE_COLOR_MAP = {
    'boundary': 'lightgreen',
    'business': '#e6005c',   # pink
    'feasible': 'white',
    'green_l': '#00ff00',
    'hospital_l': '#ff7f7e',    # light red
    'hospital_s': 'darkred',
    'business_h': '#ff0000',    # red
    'outside': 'black',
    'residential': '#ffff2d',  # yellow
    'residential_h': '#ffd380',   # light yellow
    'road': '#a3a3a3',      # grey
    'school': '#ff85c9',   # light pink
    'recreation': '#acffcf',   # light blue
}

In [35]:
roads = [
    LineString([(0,2380),(80,1720),(360,1220),(400,920),(480,400),(540,0)]),
    LineString([(640,2360),(690,2090),(740,1780),(840,1280),(910,1020),(950,890),(1010,670),(1060,540),(1200,120)]),
    LineString([(1320,2320),(1360,2120),(1420,1840),(1520,1300),(1540,1110),(1580,800),(1620,280)]),
    LineString([(1800,2340),(1860,2120),(1920,1860),(2000,1320),(1980,860),(2100,500)]),
    LineString([(2180,2360),(2360,1860),(2520,1340),(2580,1000),(2640,780)]),
    LineString([(0,2380),(640,2360),(1320,2320),(1800,2340),(2180,2360)]),
    LineString([(690,2090),(1360,2120),(1860,2120)]),
    LineString([(80,1720),(740,1780),(1420,1840),(1920,1860),(2360,1860)]),
    LineString([(360,1220),(840,1280),(1520,1300),(2000,1320),(2520,1340)]),
    LineString([(400,920),(910,1020)]),
    LineString([(950,890),(1540,1110)]),
    LineString([(1010,670),(1580,800),(1980,860),(2580,1000)]),
    LineString([(480,400),(1060,540)]),
    LineString([(540,0),(1200,120),(1620,280),(2100,500),(2640,780)])

]

In [36]:
def aggregate(lines):
    line = lines[0]
    for var in range(1, len(lines)):
        line = split(line, lines[var])
        if len(line.geoms) > 1:
            line = MultiLineString(line)
        else:
            line = line.geoms[0]
        new_line = split(lines[var], line)
        if len(new_line.geoms) > 1:
            new_line = MultiLineString(new_line)
        else:
            new_line = new_line.geoms[0]
        line = line.union(new_line)
    temp_roads = list(line.geoms)
    roads = []
    for road in temp_roads:
        if len(road.coords) == 2:
            roads.append(road)
        elif len(road.coords) > 2:
            divided_roads = [LineString([road.coords[i], road.coords[i+1]]) for i in range(len(road.coords) - 1)]
            roads.extend(divided_roads)
        else:
            raise RuntimeError('Number of segments fault!')
    roads_type = [ROAD for _ in range(len(roads))]
    intersections = []
    for road in roads:
        intersections.append(Point(road.coords[0]))
        intersections.append(Point(road.coords[1]))
    intersections = list(unary_union(intersections).geoms)
    intersections_type = [INTERSECTION for _ in range(len(intersections))]
    feasibles = list(polygonize(roads))
    feasibles_type = [FEASIBLE for _ in range(len(feasibles))]
    types = roads_type + intersections_type + feasibles_type
    geometries = roads + intersections + feasibles
    gdf = GeoDataFrame({'id': list(range(len(types))),
                        'type': types,
                        'existence': [True for _ in range(len(types))],
                        'geometry': geometries}).set_index('id')
    return gdf

In [6]:
def plot_gdf(gdf, filename=None, **kwargs):
    fig, ax = plt.subplots(1, 1, figsize=(15,15))
    gdf.plot(ax=ax, linewidth=2, markersize=15, **kwargs)
    ax.tick_params(axis='both', which='major', labelsize=15)
    if filename is not None:
        plt.savefig(filename)
    plt.show()

In [7]:
def translate_and_rescale(gdf):
    # Get the minimum x and y values from total_bounds
    x_min, y_min, x_max, y_max = gdf.total_bounds
    
    # Calculate the translation values
    x_translation = -x_min
    y_translation = -y_min
    
    # Apply the translation to the GeoDataFrame
    gdf['geometry'] = gdf.translate(xoff=x_translation, yoff=y_translation)
    
    scaling_factor = 0.1  # Scale down by a factor of 10
    
    # Create the scaling matrix
    scaling_matrix = [scaling_factor, 0, 0, scaling_factor, 0, 0]
    
    # Apply the scaling transformation to the GeoDataFrame
    gdf['geometry'] = gdf['geometry'].apply(lambda geom: affine_transform(geom, scaling_matrix))

    return gdf

In [13]:
gdf_simplified = gpd.read_file('hz_simplified2.geojson')
gdf_simplified.crs = None

In [10]:
gdf_simplified = translate_and_rescale(gdf_simplified)

In [11]:

gdf_simplified.explore()


In [59]:
gdf_poly=gdf_simplified[gdf_simplified.geom_type=='Polygon']
gdf_poly.crs=None
m = gdf_poly.explore(
    height=800,     # 增加地图高度
    width=1200,     # 增加地图宽度
    legend=True,
    style_kwds={'weight': 2.5}
)

# 获取数据边界并设置显示范围
bounds = gdf.total_bounds
m.fit_bounds([[0, 0], [3000, 2500]])  # 手动设置边界范围，确保覆盖���有点
m.save('map.html')

In [42]:
print(len(gdf_simplified[gdf_simplified.geom_type=='LineString']))
print(len(gdf_simplified[gdf_simplified.geom_type=='Point']))
print(len(gdf_simplified[gdf_simplified.geom_type=='Polygon']))

52
33
20


In [60]:
gdf_simplified.to_file('hz_simplified2.geojson', driver='GeoJSON')